In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

In [2]:
# Check GPU status
!nvidia-smi

Thu Mar 26 17:58:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

# Configuration

In [4]:
# Fix TorchDynamo bug
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["UNSLOTH_DISABLE_COMPILE"] = "1"
os.environ["UNSLOTH_DISABLE_FUSED_KERNELS"] = "1"

In [9]:
# Model configuration
MODEL_ID = 'unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit'

# Data configuration
SFT_DATA_ID = 'jxie/coco_captions'
RL_DATA_ID = 'alxxtexxr/ViRL1.25K-Seed-42'
DATA_SPLIT = 'train'
DATA_SIZE = 125
BATCH_SIZE = 10
SAVE_DATA_ID = 'alxxtexxr/BIVR-Data'

# Model

In [6]:
from unsloth import FastVisionModel
import torch

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = MODEL_ID,
    load_in_4bit = False, # False for LoRA 16bit
    use_gradient_checkpointing = 'unsloth', # True or 'unsloth' for long context
    # max_seq_length = 16384, # Must be this long for VLMs
    # fast_inference = True, # Enable vLLM fast inference
    # fast_inference = False, # Disable to fix the vLLM bug on T4
    # gpu_memory_utilization = 0.8, # Reduce if out of memory
    # torch_dtype = torch.float32,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.15: Fast Qwen2_5_Vl patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 32f60bda-9f0d-4165-8bae-867f89d6899d)')' thrown while requesting HEAD https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct/resolve/main/chat_template.jinja
Retrying in 1s [Retry 1/5].


In [10]:
from datasets import load_dataset, Dataset

def load_hf_dataset(
    data_id,
    split='train',
    train_size = 1000,
    test_size = 0,
):
    # Use streaming
    dataset_stream = load_dataset(data_id, split=split, streaming=True)

    # Manually take train_size + test_size samples
    total_size = train_size + test_size
    sliced_data = []
    
    dataset_size = 0
    unique_cocoids = set()
    
    for example in dataset_stream:
        if dataset_size >= total_size:
            break
        
        # Ensure unique cocoids if cocoid exists
        cocoid = example.get('cocoid', None)
        if cocoid is not None:
            if cocoid in unique_cocoids:
                continue
            unique_cocoids.add(cocoid)
        
        sliced_data.append(example)
        dataset_size += 1

    # Convert to regular in-memory dataset
    dataset = Dataset.from_list(sliced_data)
    
    return dataset

sft_dataset = load_hf_dataset(SFT_DATA_ID, split=DATA_SPLIT, train_size=DATA_SIZE)
rl_dataset = load_hf_dataset(RL_DATA_ID, split=DATA_SPLIT, train_size=DATA_SIZE)

print("SFT dataset:")
print(sft_dataset)
print()
print("RL dataset:")
print(rl_dataset)

Resolving data files:   0%|          | 0/182 [00:00<?, ?it/s]

SFT dataset:
Dataset({
    features: ['image', 'filename', 'cocoid', 'caption'],
    num_rows: 125
})

RL dataset:
Dataset({
    features: ['question', 'answer', 'PassRate_32BTrained', 'PassRate_7BBase', 'category', 'source', 'qid', 'image_paths', 'image'],
    num_rows: 125
})


In [11]:
import requests
from PIL import Image
from io import BytesIO

def process_image_with_url(example):
    url = example['url']

    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()

        image = Image.open(BytesIO(response.content))
        image.load()

        # Convert to RGB
        if image.mode != "RGB":
            image = image.convert("RGB")

        # Resize
        image = image.resize((512, 512), Image.LANCZOS)

    except Exception as e:
        print(f"Failed to process {url}: {e}")
        image = None

    example["decoded_image"] = image
    return example

def process_image(example):
    image_col = 'decoded_image' if 'decoded_image' in example else 'image'
    image = example[image_col]
    image = image.resize((512, 512), Image.LANCZOS)
    if image.mode != 'RGB':
        image = image.convert('RGB')
    example['decoded_image'] = image
    return example

# if 'url' in dataset.features:
#     # Load and process images from URLs
#     dataset = dataset.map(
#         process_image_with_url, 
#         num_proc=4,
#     )
# else:
#     # Resize to (512, 512) and convert to RGB
#     dataset = dataset.map(
#         process_image, 
#         # num_proc=4, # Somehow multiprocessing causes errors on T4
#     )

sft_dataset = sft_dataset.map(
    process_image,
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)
rl_dataset = rl_dataset.map(
    process_image, 
    # num_proc=4, # Somehow multiprocessing causes errors on T4
)

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

In [28]:
from datasets import concatenate_datasets

# Keep only decoded_image + add source label
sft_images = sft_dataset.select_columns(['decoded_image']).map(
    lambda x: {"source": "sft"}
)
rl_images = rl_dataset.select_columns(['decoded_image']).map(
    lambda x: {"source": "rl"}
)

# Combine
train_dataset = concatenate_datasets([sft_images, rl_images])

# Rename column
train_dataset = train_dataset.rename_column('decoded_image', 'image')

print(train_dataset)

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Map:   0%|          | 0/125 [00:00<?, ? examples/s]

Dataset({
    features: ['image', 'source'],
    num_rows: 250
})


In [16]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(MODEL_ID)

preprocessor_config.json:   0%|          | 0.00/791 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/935 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

In [29]:
import math
import os
from pathlib import Path
import numpy as np
from huggingface_hub import upload_file
from sklearn.preprocessing import StandardScaler

train_size = len(train_dataset)
save_dir = Path(
    f"model_{MODEL_ID.replace('/', '_')}"
    f".data_sft_{SFT_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
    f".data_rl_{RL_DATA_ID.replace('/', '_')}_{DATA_SIZE}"
)
os.makedirs(save_dir, exist_ok=True)

scaler = StandardScaler()

for i in range(math.ceil(train_size / BATCH_SIZE)):
    start_idx = i * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, train_size)
    range_tag = f'{start_idx}-{end_idx-1}'
    
    print("="*128)
    print("Range:", range_tag)
    print("="*128)
    
    inputs = processor.image_processor(images=list(train_dataset['image'][start_idx:end_idx]), return_tensors='pt')
    pixel_values = inputs['pixel_values'].to(model.device)
    image_grid_thw = inputs['image_grid_thw'].to(model.device)
    
    print("Pixel values shape:", pixel_values.shape)
    print("Image grid shape:", image_grid_thw.shape)
    
    with torch.no_grad():
        visual_embs = model.visual(pixel_values, image_grid_thw)

    print("Visual embeddings shape:", visual_embs.shape)
    print()
    
    # batch = visual_embs.float() # Convert to float32 for statistics calculation -> (N, 2048)

    # # Statistics for Standardization
    # batch_n = batch.shape[0]
    # batch_sum = batch.sum(dim=0)
    # batch_sum_sq = (batch**2).sum(dim=0)

    # # Statistics for Min-Max Normalization
    # batch_min = batch.min(dim=0).values
    # batch_max = batch.max(dim=0).values

    # print("Batch size:", batch_n)
    # print("Batch sum average:", batch_sum.mean())
    # print("Batch sum squared average:", batch_sum_sq.mean())
    # print("Batch min average:", batch_min.mean())
    # print("Batch max average:", batch_max.mean())
    # print()

    # Partial fit the scaler with current batch
    scaler.partial_fit(visual_embs.detach().cpu().numpy())

    # Save vision data and statistics
    range_tag = f'{str(start_idx).zfill(len(str(DATA_SIZE)))}-{str(end_idx-1).zfill(len(str(DATA_SIZE)))}'
    vision_data_path = save_dir / f'vision_data.{range_tag}.npz'
    # stats_path = save_dir / f'{range_tag}.stats.npz'
    
    np.savez(
        vision_data_path,
        pixel_values=pixel_values.cpu().numpy(),
        image_grid_thw=image_grid_thw.cpu().numpy(),
        visual_embs=visual_embs.cpu().numpy()
    )
    # np.savez(
    #     stats_path,
    #     batch_n=batch_n,
    #     batch_sum=batch_sum.cpu().numpy(),
    #     batch_sum_sq=batch_sum_sq.cpu().numpy(),
    #     batch_min=batch_min.cpu().numpy(),
    #     batch_max=batch_max.cpu().numpy()
    # )

    print("Vision data saved to:", vision_data_path)
    # print("Statistics saved to:", stats_path)

    upload_file(
        path_or_fileobj=str(vision_data_path),
        path_in_repo=str(vision_data_path),
        repo_id=SAVE_DATA_ID,
        repo_type='dataset',
    )
    # upload_file(
    #     path_or_fileobj=str(stats_path),
    #     path_in_repo=str(stats_path),
    #     repo_id=SAVE_DATA_ID,
    #     repo_type='dataset',
    # )

model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125
Range: 0-9
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125/vision_data.000-009.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5/vision_data.000-009.npz:  28%|##8       | 23.9MB / 84.2MB            

Range: 10-19
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125/vision_data.010-019.npz


...

  ...5/vision_data.230-239.npz:   2%|2         | 1.84MB / 84.2MB            

Range: 240-249
Pixel values shape: torch.Size([12960, 1176])
Image grid shape: torch.Size([10, 3])
Visual embeddings shape: torch.Size([3240, 3584])

Vision data saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125/vision_data.240-249.npz


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5/vision_data.240-249.npz:   3%|2         | 2.49MB / 84.2MB            

In [30]:
import joblib

# Save the scaler
scaler_path = save_dir / 'scaler.pkl'
joblib.dump(scaler, scaler_path)

print("Scaler saved to:", scaler_path)

upload_file(
    path_or_fileobj=str(scaler_path),
    path_in_repo=str(scaler_path),
    repo_id='alxxtexxr/BIVR-Data',
    repo_type='dataset',
)

Scaler saved to: model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125/scaler.pkl


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...5K-Seed-42_125/scaler.pkl: 100%|##########| 86.6kB / 86.6kB            

CommitInfo(commit_url='https://huggingface.co/datasets/alxxtexxr/BIVR-Data/commit/fa63362f3e87b96599e418d6696f823749725939', commit_message='Upload model_unsloth_Qwen2.5-VL-7B-Instruct-bnb-4bit.data_sft_jxie_coco_captions_125.data_rl_alxxtexxr_ViRL1.25K-Seed-42_125/scaler.pkl with huggingface_hub', commit_description='', oid='fa63362f3e87b96599e418d6696f823749725939', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/alxxtexxr/BIVR-Data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='alxxtexxr/BIVR-Data'), pr_revision=None, pr_num=None)